# 02 資料前處理

本 Notebook 依據 Jain et al. (2024) 之 Methods，重現 SPARCS 2016 住院資料集之前處理流程。本章目標是將原始資料整理為後續機器學習模型可使用的分析資料集，並完整記錄資料清理過程，以確保研究流程具有可重現性。

本章包含以下流程：

1. 載入套件與設定路徑
2. 讀取原始資料
3. 檢查資料維度與欄位
4. 缺失值檢查
5. 建立前處理紀錄表
6. 建立紀錄函數
7. 移除缺失值
8. 處理未知入院型態
9. 處理 Payment Typology 欄位
10. 分割 Newborn 與 Non-newborn 資料
11. 移除 Non-newborn 資料中的 Birth Weight 欄位
12. 移除資料洩漏變項
13. 移除冗餘欄位
14. 儲存前處理後資料
15. 前處理結果驗證

## 2.1 載入套件與設定路徑

本節載入資料前處理所需之 R 套件，並設定 SPARCS 2016 原始資料檔案路徑。讀取資料前先確認檔案是否存在，以避免後續分析因路徑錯誤而中斷。


In [18]:
#--------------------------------------------------
# 2.1 載入套件與設定路徑
#--------------------------------------------------

library(data.table)
library(dplyr)
library(tidyr)
library(stringr)
library(skimr)
library(janitor)

raw_file <- "../data/raw/Hospital_Inpatient_Discharges_(SPARCS_De-Identified)__2016_20260630.csv"

file.exists(raw_file)

[1] TRUE

## 2.2 讀取原始資料

本研究使用 New York State SPARCS 2016 年公開住院資料集。資料格式為 CSV，原始資料應包含 2,343,429 筆住院紀錄與 34 個欄位。

本節先讀取完整原始資料，並確認資料筆數與欄位數是否與原論文描述一致。

In [20]:
#--------------------------------------------------
# 2.2 讀取原始資料
#--------------------------------------------------

df_raw <- fread(raw_file)

dim(df_raw)

[1] 2343429      34

In [21]:
names(df_raw)

[1] "Health Service Area"                 "Hospital County"                    
 [3] "Operating Certificate Number"        "Facility Id"                        
 [5] "Facility Name"                       "Age Group"                          
 [7] "Zip Code - 3 digits"                 "Gender"                             
 [9] "Race"                                "Ethnicity"                          
[11] "Length of Stay"                      "Type of Admission"                  
[13] "Patient Disposition"                 "Discharge Year"                     
[15] "CCS Diagnosis Code"                  "CCS Diagnosis Description"          
[17] "CCS Procedure Code"                  "CCS Procedure Description"          
[19] "APR DRG Code"                        "APR DRG Description"                
[21] "APR MDC Code"                        "APR MDC Description"                
[23] "APR Severity of Illness Code"        "APR Severity of Illness Description"
[25] "APR Risk of Mortality"               "APR Medical Surgical Description"   
[27] "Payment Typology 1"                  "Payment Typology 2"                 
[29] "Payment Typology 3"                  "Birth Weight"                       
[31] "Abortion Edit Indicator"             "Emergency Department Indicator"     
[33] "Total Charges"                       "Total Costs"

## 2.3 檢查資料維度與欄位

讀取資料後，先檢查資料筆數、欄位數與欄位名稱。根據 Jain et al. (2024)，SPARCS 2016 原始資料包含 2,343,429 筆紀錄與 34 個欄位。因此，本節用於確認目前讀入之資料是否與原論文描述一致。

In [22]:
#--------------------------------------------------
# 2.3 檢查資料維度與欄位
#--------------------------------------------------

dim(df_raw)

[1] 2343429      34

In [23]:
names(df_raw)

[1] "Health Service Area"                 "Hospital County"                    
 [3] "Operating Certificate Number"        "Facility Id"                        
 [5] "Facility Name"                       "Age Group"                          
 [7] "Zip Code - 3 digits"                 "Gender"                             
 [9] "Race"                                "Ethnicity"                          
[11] "Length of Stay"                      "Type of Admission"                  
[13] "Patient Disposition"                 "Discharge Year"                     
[15] "CCS Diagnosis Code"                  "CCS Diagnosis Description"          
[17] "CCS Procedure Code"                  "CCS Procedure Description"          
[19] "APR DRG Code"                        "APR DRG Description"                
[21] "APR MDC Code"                        "APR MDC Description"                
[23] "APR Severity of Illness Code"        "APR Severity of Illness Description"
[25] "APR Risk of Mortality"               "APR Medical Surgical Description"   
[27] "Payment Typology 1"                  "Payment Typology 2"                 
[29] "Payment Typology 3"                  "Birth Weight"                       
[31] "Abortion Edit Indicator"             "Emergency Department Indicator"     
[33] "Total Charges"                       "Total Costs"

In [24]:
glimpse(df_raw)

Rows: 2,343,429
Columns: 34
$ `Health Service Area`                 <chr> "Western NY", "Western NY", "Wes…
$ `Hospital County`                     <chr> "Allegany", "Allegany", "Allegan…
$ `Operating Certificate Number`        <int> 226700, 226700, 226700, 226700, …
$ `Facility Id`                         <chr> "37", "37", "37", "37", "37", "3…
$ `Facility Name`                       <chr> "Cuba Memorial Hospital Inc", "C…
$ `Age Group`                           <chr> "70 or Older", "30 to 49", "50 t…
$ `Zip Code - 3 digits`                 <chr> "147", "147", "147", "147", "147…
$ Gender                                <chr> "F", "M", "F", "M", "M", "M", "M…
$ Race                                  <chr> "White", "White", "White", "Whit…
$ Ethnicity                             <chr> "Not Span/Hispanic", "Not Span/H…
$ `Length of Stay`                      <chr> "3", "2", "7", "4", "5", "1", "2…
$ `Type of Admission`                   <chr> "Urgent", "Elective", "Urgent", …
$ `Patient D

## 2.4 缺失值檢查

在正式進行資料前處理前，需先檢查各欄位缺失值分布情形。原論文指出，2016 年 SPARCS 原始資料中共有 36,280 筆紀錄存在缺失值，約占全部資料 1.55%，並於後續分析中排除。

本研究使用目前公開取得之 SPARCS 2016 資料版本，因此重新計算缺失值數量與比例，以確認目前資料版本與原論文之差異。

In [25]:
#--------------------------------------------------
# 2.4 缺失值檢查
#--------------------------------------------------

missing_summary <- data.frame(
  Variable = names(df_raw),
  Missing = colSums(is.na(df_raw))
) %>%
  mutate(
    Percent = round(Missing / nrow(df_raw) * 100, 3)
  ) %>%
  arrange(desc(Missing))

missing_summary

,Variable,Missing,Percent
,<chr>,<dbl>,<dbl>
Operating Certificate Number,Operating Certificate Number,5325,0.227
Health Service Area,Health Service Area,0,0.000
Hospital County,Hospital County,0,0.000
Facility Id,Facility Id,0,0.000
Facility Name,Facility Name,0,0.000
Age Group,Age Group,0,0.000
Zip Code - 3 digits,Zip Code - 3 digits,0,0.000
Gender,Gender,0,0.000
Race,Race,0,0.000


In [26]:
missing_rows <- sum(!complete.cases(df_raw))
missing_percent <- missing_rows / nrow(df_raw) * 100

missing_rows
missing_percent

[1] 5325

[1] 0.2272311

### 2.4 結果說明

目前資料版本中，含有缺失值的資料筆數少於原論文所報告之 36,280 筆。缺失值主要集中於 `Operating Certificate Number` 欄位，而 `Payment Typology 2` 與 `Payment Typology 3` 並未出現原論文所描述的大量缺失值。

此差異可能與 SPARCS 公開資料後續版本更新、欄位補值或隱私保護處理有關。因此，本研究後續將重現原論文之前處理流程，但同時保留目前資料版本之實際處理紀錄。

## 2.5 建立前處理紀錄表

為了提高資料處理流程的可追蹤性，本研究建立前處理紀錄表，用來記錄每一個資料清理步驟前後的樣本數與欄位數變化。

此紀錄表可協助確認各步驟是否符合原論文 Methods 所描述之流程，並可作為後續報告與 GitHub 專案中說明資料前處理過程的依據。

In [27]:
#--------------------------------------------------
# 2.5 建立前處理紀錄表
#--------------------------------------------------

preprocess_log <- data.frame(
  Step = "原始資料",
  Rows_Before = NA,
  Rows_After = nrow(df_raw),
  Rows_Removed = NA,
  Columns_Before = NA,
  Columns_After = ncol(df_raw),
  Columns_Removed = NA
)

preprocess_log

Step,Rows_Before,Rows_After,Rows_Removed,Columns_Before,Columns_After,Columns_Removed
<chr>,<lgl>,<int>,<lgl>,<lgl>,<int>,<lgl>
原始資料,NA,2343429,NA,NA,34,NA


## 2.6 建立紀錄函數

為了避免每一個前處理步驟都手動建立紀錄，本研究建立一個自訂函數 `add_log()`。此函數會自動計算每一步處理前後的資料筆數、欄位數，以及移除的筆數與欄位數。

In [28]:
#--------------------------------------------------
# 2.6 建立紀錄函數
#--------------------------------------------------

add_log <- function(log_table,
                    step_name,
                    rows_before,
                    rows_after,
                    cols_before,
                    cols_after) {
  
  new_row <- data.frame(
    Step = step_name,
    Rows_Before = rows_before,
    Rows_After = rows_after,
    Rows_Removed = rows_before - rows_after,
    Columns_Before = cols_before,
    Columns_After = cols_after,
    Columns_Removed = cols_before - cols_after
  )
  
  rbind(log_table, new_row)
}

## 2.7 移除缺失值

根據 Jain et al. (2024) 的前處理流程，作者移除所有包含缺失值的觀測值。雖然本研究使用之目前公開資料版本缺失值數量較少，但為了忠實重現原論文的資料處理流程，本研究仍移除所有包含缺失值的資料列。

此步驟完成後，資料筆數會減少，但欄位數應維持不變。

In [29]:
#--------------------------------------------------
# 2.7 移除缺失值
#--------------------------------------------------

rows_before <- nrow(df_raw)
cols_before <- ncol(df_raw)

df_clean <- df_raw %>%
  drop_na()

rows_after <- nrow(df_clean)
cols_after <- ncol(df_clean)

preprocess_log <- add_log(
  log_table = preprocess_log,
  step_name = "移除缺失值",
  rows_before = rows_before,
  rows_after = rows_after,
  cols_before = cols_before,
  cols_after = cols_after
)

preprocess_log

Step,Rows_Before,Rows_After,Rows_Removed,Columns_Before,Columns_After,Columns_Removed
<chr>,<int>,<int>,<int>,<int>,<int>,<int>
原始資料,NA,2343429,NA,NA,34,NA
移除缺失值,2343429,2338104,5325,34,34,0


### 2.7 結果說明

移除缺失值後，資料筆數下降，但欄位數維持不變。此結果表示本步驟僅移除包含缺失值的觀測值，並未刪除任何變項。

由於目前資料版本的缺失值數量少於原論文所報告之 36,280 筆，因此後續結果可能與原論文最終樣本數略有差異。此差異將於討論章節中作為公開資料版本更新造成之重現性差異進行說明。

## 2.8 處理未知入院型態

根據 Jain et al. (2024)，作者於缺失值處理完成後，進一步排除 **Type of Admission** 為 **Unknown** 之住院紀錄。論文指出此類資料約占全部樣本 0.02%，因此直接予以移除，以避免未知入院型態對後續模型建立造成影響。

本研究依照相同流程，檢查目前資料集中 Unknown 類別之數量，並移除相關觀測值。

In [30]:
#--------------------------------------------------
# 2.8 處理未知入院型態
#--------------------------------------------------

table(df_clean$`Type of Admission`)


     Elective     Emergency       Newborn Not Available        Trauma 
       440587       1494930        224573           432          6619 
       Urgent 
       170963 

In [31]:
rows_before <- nrow(df_clean)
cols_before <- ncol(df_clean)

df_clean <- df_clean %>%
  filter(`Type of Admission` != "Unknown")

rows_after <- nrow(df_clean)
cols_after <- ncol(df_clean)

preprocess_log <- add_log(
  log_table = preprocess_log,
  step_name = "移除 Unknown Admission",
  rows_before = rows_before,
  rows_after = rows_after,
  cols_before = cols_before,
  cols_after = cols_after
)

preprocess_log

Step,Rows_Before,Rows_After,Rows_Removed,Columns_Before,Columns_After,Columns_Removed
<chr>,<int>,<int>,<int>,<int>,<int>,<int>
原始資料,NA,2343429,NA,NA,34,NA
移除缺失值,2343429,2338104,5325,34,34,0
移除 Unknown Admission,2338104,2338104,0,34,34,0


### 2.8 結果說明

完成此步驟後，所有入院型態為 Unknown 之觀測值均已移除。由於 Unknown 類別所占比例極低，因此對整體資料量影響有限，但可提升後續分析資料之完整性。

## 2.9 處理 Payment Typology 欄位

原論文指出，**Payment Typology 2** 與 **Payment Typology 3** 兩個欄位具有超過 50% 的缺失值，因此作者並未移除欄位，而是將缺失值統一以字串 **"None"** 表示。

本研究重新檢查目前公開資料版本後發現，此兩個欄位已無缺失值，因此本步驟僅驗證目前資料狀況，並保留原始內容。

In [32]:
colSums(is.na(
  df_clean[, c("Payment Typology 2",
               "Payment Typology 3")]
))

Payment Typology 2 Payment Typology 3 
                 0                  0

In [33]:
df_clean <- df_clean %>%
  mutate(
    `Payment Typology 2` =
      replace_na(`Payment Typology 2`, "None"),
    `Payment Typology 3` =
      replace_na(`Payment Typology 3`, "None")
  )

## 2.10 分割 Newborn 與 Non-newborn 資料

原論文指出，約 10% 的資料屬於 newborn 類別。由於 **Birth Weight** 欄位主要適用於新生兒，而非新生兒資料中 Birth Weight 多為 0，因此作者將資料分割為 newborn 與 non-newborn 兩個子資料集，並分別建立模型。

本研究依照原論文流程，根據 **APR DRG Description** 是否包含 newborn 或 neonate 相關描述，將資料分為新生兒與非新生兒資料集。

In [34]:
#--------------------------------------------------
# 2.10 分割 Newborn 與 Non-newborn 資料
#--------------------------------------------------

df_newborn <- df_clean %>%
  filter(str_detect(
    str_to_lower(`APR DRG Description`),
    "newborn|neonate"
  ))

df_non_newborn <- df_clean %>%
  filter(!str_detect(
    str_to_lower(`APR DRG Description`),
    "newborn|neonate"
  ))

nrow(df_newborn)
nrow(df_non_newborn)
nrow(df_newborn) + nrow(df_non_newborn)
nrow(df_clean)

[1] 229290

[1] 2108814

[1] 2338104

[1] 2338104

### 2.10 結果說明

完成資料分割後，newborn 與 non-newborn 兩個資料集的筆數加總應等於分割前的資料筆數，表示資料分割過程未遺漏或重複觀測值。

此分割方式可讓後續模型分別處理新生兒與非新生兒族群，符合原論文對 Birth Weight 變項適用性的考量。

## 2.11 移除 Non-newborn 資料中的 Birth Weight 欄位

原論文指出，Birth Weight 變項主要用於 newborn 資料，而在 non-newborn 資料中其值多為 0，缺乏實質預測意義。因此，作者在 non-newborn 資料集中移除 Birth Weight 欄位。

本研究依照相同邏輯，僅於 non-newborn 資料集中移除 Birth Weight 欄位；newborn 資料則保留此欄位。

In [35]:
#--------------------------------------------------
# 2.11 移除 Non-newborn 資料中的 Birth Weight 欄位
#--------------------------------------------------

rows_before <- nrow(df_non_newborn)
cols_before <- ncol(df_non_newborn)

df_non_newborn <- df_non_newborn %>%
  select(-`Birth Weight`)

rows_after <- nrow(df_non_newborn)
cols_after <- ncol(df_non_newborn)

preprocess_log <- add_log(
  log_table = preprocess_log,
  step_name = "Non-newborn 移除 Birth Weight",
  rows_before = rows_before,
  rows_after = rows_after,
  cols_before = cols_before,
  cols_after = cols_after
)

preprocess_log

Step,Rows_Before,Rows_After,Rows_Removed,Columns_Before,Columns_After,Columns_Removed
<chr>,<int>,<int>,<int>,<int>,<int>,<int>
原始資料,NA,2343429,NA,NA,34,NA
移除缺失值,2343429,2338104,5325,34,34,0
移除 Unknown Admission,2338104,2338104,0,34,34,0
Non-newborn 移除 Birth Weight,2108814,2108814,0,34,33,1


### 2.11 結果說明

完成此步驟後，non-newborn 資料集的欄位數減少 1 個，而資料筆數維持不變。此結果符合原論文描述：Birth Weight 僅保留於 newborn 模型中，不納入 non-newborn 模型。

## 2.12 移除資料洩漏變項

原論文指出，**Total Charges** 與 **Total Costs** 通常會隨住院天數增加而上升，因此與研究目標變項 **Length of Stay** 存在高度直接關係。若將這兩個欄位納入模型，可能造成資料洩漏（Data Leakage），使模型表現被高估。

因此，作者於建模前移除 **Total Charges** 與 **Total Costs**。本研究依照相同原則，分別於 newborn 與 non-newborn 資料集中移除此兩個欄位。

In [36]:
#--------------------------------------------------
# 2.12 移除資料洩漏變項
#--------------------------------------------------

leakage_vars <- c("Total Charges", "Total Costs")

rows_before <- nrow(df_newborn)
cols_before <- ncol(df_newborn)

df_newborn <- df_newborn %>%
  select(-all_of(leakage_vars))

rows_after <- nrow(df_newborn)
cols_after <- ncol(df_newborn)

preprocess_log <- add_log(
  log_table = preprocess_log,
  step_name = "Newborn 移除資料洩漏變項",
  rows_before = rows_before,
  rows_after = rows_after,
  cols_before = cols_before,
  cols_after = cols_after
)

rows_before <- nrow(df_non_newborn)
cols_before <- ncol(df_non_newborn)

df_non_newborn <- df_non_newborn %>%
  select(-all_of(leakage_vars))

rows_after <- nrow(df_non_newborn)
cols_after <- ncol(df_non_newborn)

preprocess_log <- add_log(
  log_table = preprocess_log,
  step_name = "Non-newborn 移除資料洩漏變項",
  rows_before = rows_before,
  rows_after = rows_after,
  cols_before = cols_before,
  cols_after = cols_after
)

preprocess_log

Step,Rows_Before,Rows_After,Rows_Removed,Columns_Before,Columns_After,Columns_Removed
<chr>,<int>,<int>,<int>,<int>,<int>,<int>
原始資料,NA,2343429,NA,NA,34,NA
移除缺失值,2343429,2338104,5325,34,34,0
移除 Unknown Admission,2338104,2338104,0,34,34,0
Non-newborn 移除 Birth Weight,2108814,2108814,0,34,33,1
Newborn 移除資料洩漏變項,229290,229290,0,34,32,2
Non-newborn 移除資料洩漏變項,2108814,2108814,0,33,31,2


### 2.12 結果說明

完成此步驟後，newborn 與 non-newborn 資料集的欄位數皆減少 2 個，而資料筆數維持不變。此結果表示本步驟僅移除可能造成資料洩漏的欄位，未排除任何觀測值。

移除 Total Charges 與 Total Costs 可避免模型利用與住院天數高度相關、且通常在住院結束後才完整得知的資訊進行預測，較符合入院早期預測住院天數的研究目的。

## 2.13 移除冗餘欄位

依據原論文 Methods，除資料洩漏變項外，作者亦移除數個與模型建立無直接關聯或資訊重複之欄位。

其中 **Discharge Year** 為固定年度資訊，對於單一年度資料集不具區辨能力；**Abortion Edit Indicator** 與研究目標無直接關聯，因此予以移除。

此外，資料集中部分文字描述欄位已具有相對應之數值代碼，例如 **CCS Diagnosis Description** 對應 **CCS Diagnosis Code**，**APR DRG Description** 對應 **APR DRG Code**。因此，本研究保留代碼欄位並移除重複之文字描述欄位，以降低模型輸入維度並維持與原論文一致之前處理流程。

本節移除下列冗餘欄位：

- Discharge Year
- Abortion Edit Indicator
- CCS Diagnosis Description
- CCS Procedure Description
- APR DRG Description
- APR MDC Description
- APR Severity of Illness Description


In [39]:
#--------------------------------------------------
# 2.13 移除冗餘欄位
#--------------------------------------------------

redundant_vars <- c(
  "Discharge Year",
  "Abortion Edit Indicator",
  "CCS Diagnosis Description",
  "CCS Procedure Description",
  "APR DRG Description",
  "APR MDC Description",
  "APR Severity of Illness Description"
)

#-----------------------------
# Newborn
#-----------------------------

rows_before <- nrow(df_newborn)
cols_before <- ncol(df_newborn)

remove_cols <- intersect(redundant_vars, names(df_newborn))

df_newborn <- df_newborn %>%
  select(-all_of(remove_cols))

rows_after <- nrow(df_newborn)
cols_after <- ncol(df_newborn)

preprocess_log <- add_log(
  log_table = preprocess_log,
  step_name = "Newborn 移除冗餘欄位",
  rows_before = rows_before,
  rows_after = rows_after,
  cols_before = cols_before,
  cols_after = cols_after
)

#-----------------------------
# Non-newborn
#-----------------------------

rows_before <- nrow(df_non_newborn)
cols_before <- ncol(df_non_newborn)

remove_cols <- intersect(redundant_vars, names(df_non_newborn))

df_non_newborn <- df_non_newborn %>%
  select(-all_of(remove_cols))

rows_after <- nrow(df_non_newborn)
cols_after <- ncol(df_non_newborn)

preprocess_log <- add_log(
  log_table = preprocess_log,
  step_name = "Non-newborn 移除冗餘欄位",
  rows_before = rows_before,
  rows_after = rows_after,
  cols_before = cols_before,
  cols_after = cols_after
)

#-----------------------------
# 顯示前處理紀錄
#-----------------------------

preprocess_log

Step,Rows_Before,Rows_After,Rows_Removed,Columns_Before,Columns_After,Columns_Removed
<chr>,<int>,<int>,<int>,<int>,<int>,<int>
原始資料,NA,2343429,NA,NA,34,NA
移除缺失值,2343429,2338104,5325,34,34,0
移除 Unknown Admission,2338104,2338104,0,34,34,0
Non-newborn 移除 Birth Weight,2108814,2108814,0,34,33,1
Newborn 移除資料洩漏變項,229290,229290,0,34,32,2
Non-newborn 移除資料洩漏變項,2108814,2108814,0,33,31,2
Newborn 移除描述性欄位,229290,229290,0,32,27,5
Non-newborn 移除描述性欄位,2108814,2108814,0,31,26,5
Newborn 移除冗餘欄位,229290,229290,0,27,25,2


### 2.13 結果說明

完成此步驟後，newborn 與 non-newborn 資料集均移除 7 個冗餘欄位，資料筆數維持不變。

保留數值代碼而移除文字描述，可避免相同資訊重複輸入模型，同時降低資料維度；此外，固定年度資訊與研究無關欄位亦一併移除，使後續模型建立流程與原論文一致。

## 2.14 儲存前處理後資料

完成資料清理與分割後，本研究將 newborn 與 non-newborn 兩個資料集分別儲存至 `data/processed/` 資料夾，作為後續模型建立與分析使用。

本研究同時儲存 R 原生格式 `.rds` 與通用格式 `.csv`。其中 `.rds` 可保留 R 物件型態並加快後續讀取速度；`.csv` 則方便跨軟體檢視與 GitHub 專案展示。

In [40]:
#--------------------------------------------------
# 2.14 儲存前處理後資料
#--------------------------------------------------

processed_dir <- "../data/processed"

if (!dir.exists(processed_dir)) {
  dir.create(processed_dir, recursive = TRUE)
}

saveRDS(df_newborn,
        file = file.path(processed_dir, "df_newborn.rds"))

saveRDS(df_non_newborn,
        file = file.path(processed_dir, "df_non_newborn.rds"))

fwrite(df_newborn,
       file = file.path(processed_dir, "df_newborn.csv"))

fwrite(df_non_newborn,
       file = file.path(processed_dir, "df_non_newborn.csv"))

fwrite(preprocess_log,
       file = file.path(processed_dir, "preprocess_log.csv"))

list.files(processed_dir)

[1] "df_newborn.csv"     "df_newborn.rds"     "df_non_newborn.csv"
[4] "df_non_newborn.rds" "preprocess_log.csv"

### 2.14 結果說明

本步驟完成後，前處理後之 newborn 與 non-newborn 資料集已成功輸出至 `data/processed/` 資料夾。後續 Notebook 將優先使用 `.rds` 檔案，以提升讀取效率並保留資料型態。

## 2.15 前處理結果驗證

完成所有資料前處理步驟後，本節檢查兩個分析資料集之資料筆數、欄位數及前處理紀錄，以確認整個資料清理流程已正確完成。

此外，本研究將目前資料處理結果與原論文進行比較，作為後續重現性（Reproducibility）分析之依據。

In [41]:
#--------------------------------------------------
# 2.15 前處理結果驗證
#--------------------------------------------------

data.frame(
  Dataset = c("Newborn", "Non-newborn"),
  Rows = c(nrow(df_newborn),
           nrow(df_non_newborn)),
  Columns = c(ncol(df_newborn),
              ncol(df_non_newborn))
)

Dataset,Rows,Columns
<chr>,<int>,<int>
Newborn,229290,25
Non-newborn,2108814,24


In [42]:
names(df_newborn)

names(df_non_newborn)

[1] "Health Service Area"              "Hospital County"                 
 [3] "Operating Certificate Number"     "Facility Id"                     
 [5] "Facility Name"                    "Age Group"                       
 [7] "Zip Code - 3 digits"              "Gender"                          
 [9] "Race"                             "Ethnicity"                       
[11] "Length of Stay"                   "Type of Admission"               
[13] "Patient Disposition"              "CCS Diagnosis Code"              
[15] "CCS Procedure Code"               "APR DRG Code"                    
[17] "APR MDC Code"                     "APR Severity of Illness Code"    
[19] "APR Risk of Mortality"            "APR Medical Surgical Description"
[21] "Payment Typology 1"               "Payment Typology 2"              
[23] "Payment Typology 3"               "Birth Weight"                    
[25] "Emergency Department Indicator"

[1] "Health Service Area"              "Hospital County"                 
 [3] "Operating Certificate Number"     "Facility Id"                     
 [5] "Facility Name"                    "Age Group"                       
 [7] "Zip Code - 3 digits"              "Gender"                          
 [9] "Race"                             "Ethnicity"                       
[11] "Length of Stay"                   "Type of Admission"               
[13] "Patient Disposition"              "CCS Diagnosis Code"              
[15] "CCS Procedure Code"               "APR DRG Code"                    
[17] "APR MDC Code"                     "APR Severity of Illness Code"    
[19] "APR Risk of Mortality"            "APR Medical Surgical Description"
[21] "Payment Typology 1"               "Payment Typology 2"              
[23] "Payment Typology 3"               "Emergency Department Indicator"

In [43]:
preprocess_log

Step,Rows_Before,Rows_After,Rows_Removed,Columns_Before,Columns_After,Columns_Removed
<chr>,<int>,<int>,<int>,<int>,<int>,<int>
原始資料,NA,2343429,NA,NA,34,NA
移除缺失值,2343429,2338104,5325,34,34,0
移除 Unknown Admission,2338104,2338104,0,34,34,0
Non-newborn 移除 Birth Weight,2108814,2108814,0,34,33,1
Newborn 移除資料洩漏變項,229290,229290,0,34,32,2
Non-newborn 移除資料洩漏變項,2108814,2108814,0,33,31,2
Newborn 移除描述性欄位,229290,229290,0,32,27,5
Non-newborn 移除描述性欄位,2108814,2108814,0,31,26,5
Newborn 移除冗餘欄位,229290,229290,0,27,25,2


In [44]:
colSums(is.na(df_newborn))

colSums(is.na(df_non_newborn))

Health Service Area                  Hospital County 
                               0                                0 
    Operating Certificate Number                      Facility Id 
                               0                                0 
                   Facility Name                        Age Group 
                               0                                0 
             Zip Code - 3 digits                           Gender 
                               0                                0 
                            Race                        Ethnicity 
                               0                                0 
                  Length of Stay                Type of Admission 
                               0                                0 
             Patient Disposition               CCS Diagnosis Code 
                               0                                0 
              CCS Procedure Code                     APR DRG Code 
                               0                                0 
                    APR MDC Code     APR Severity of Illness Code 
                               0                                0 
           APR Risk of Mortality APR Medical Surgical Description 
                               0                                0 
              Payment Typology 1               Payment Typology 2 
                               0                                0 
              Payment Typology 3                     Birth Weight 
                               0                                0 
  Emergency Department Indicator 
                               0

Health Service Area                  Hospital County 
                               0                                0 
    Operating Certificate Number                      Facility Id 
                               0                                0 
                   Facility Name                        Age Group 
                               0                                0 
             Zip Code - 3 digits                           Gender 
                               0                                0 
                            Race                        Ethnicity 
                               0                                0 
                  Length of Stay                Type of Admission 
                               0                                0 
             Patient Disposition               CCS Diagnosis Code 
                               0                                0 
              CCS Procedure Code                     APR DRG Code 
                               0                                0 
                    APR MDC Code     APR Severity of Illness Code 
                               0                                0 
           APR Risk of Mortality APR Medical Surgical Description 
                               0                                0 
              Payment Typology 1               Payment Typology 2 
                               0                                0 
              Payment Typology 3   Emergency Department Indicator 
                               0                                0

In [45]:
c(
  "Total Charges",
  "Total Costs",
  "Discharge Year",
  "Abortion Edit Indicator",
  "CCS Diagnosis Description",
  "CCS Procedure Description",
  "APR DRG Description",
  "APR MDC Description",
  "APR Severity of Illness Description",
  "Birth Weight"
) %in% names(df_non_newborn)

[1] FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE FALSE

In [46]:
"Birth Weight" %in% names(df_newborn)

[1] TRUE

### 2.15 結果說明

完成所有資料前處理步驟後，newborn 與 non-newborn 資料集均已建立完成，並完成缺失值處理、未知入院型態移除、資料洩漏變項移除及冗餘欄位刪除。

檢查結果顯示，前處理後資料集中已不存在缺失值，且所有應移除之欄位均已成功刪除；Birth Weight 則僅保留於 newborn 資料集，符合原論文之資料處理流程。

由於目前使用之 SPARCS 2016 公開資料版本與原論文使用版本存在差異，因此最終資料筆數可能與原論文略有不同。本研究保留此差異，並於後續討論章節說明公開資料版本更新對研究重現性之影響。